# **Data Pipeline Example**

* All data source names have been cleansed to produce anonymity.
* This pipeline was generated by myself and serves to clean up raw data inputs, merge them on a single grain, and output a usable table for visualization (Dashboard).

**Source 1: Google Ads**

In [ ]:
CREATE OR REPLACE TABLE `XXXX` AS
WITH source_data AS (
  SELECT
    SAFE_CAST(Date AS DATE) AS date,
    CAST(Campaign_ID__Google_Ads AS STRING) AS campaign_id,
    CAST(Campaign AS STRING) AS campaign_name,
    CAST(Traffic_source AS STRING) AS traffic_source,
    CAST(Media_type AS STRING) AS media_type,
    CAST(Paid__Organic AS STRING) AS paid_organic,
    CAST(Currency AS STRING) AS currency,

    SAFE_CAST(Cost AS NUMERIC) AS google_ads_cost,
    SAFE_CAST(Clicks AS NUMERIC) AS google_ads_clicks,
    SAFE_CAST(Impressions AS NUMERIC) AS google_ads_impressions,
    SAFE_CAST(All_Conv__Value__Purchase__Google_Ads AS NUMERIC) AS google_ads_conversion_value,
    SAFE_CAST(Conversions__Google_Ads AS NUMERIC) AS google_ads_conversions
  FROM `XXXXX`
),

cleaned AS (
  SELECT
    date,
    campaign_id,
    campaign_name,
    traffic_source,
    media_type,
    paid_organic,
    currency,

    COALESCE(google_ads_cost, 0) AS google_ads_cost,
    COALESCE(google_ads_clicks, 0) AS google_ads_clicks,
    COALESCE(google_ads_impressions, 0) AS google_ads_impressions,
    COALESCE(google_ads_conversion_value, 0) AS google_ads_conversion_value,
    COALESCE(google_ads_conversions, 0) AS google_ads_conversions
  FROM source_data_XXXX
)

SELECT
  date,
  campaign_id,
  ANY_VALUE(campaign_name) AS campaign_name,
  ANY_VALUE(traffic_source) AS traffic_source,
  ANY_VALUE(media_type) AS media_type,
  ANY_VALUE(paid_organic) AS paid_organic,
  ANY_VALUE(currency) AS currency,

  SUM(google_ads_cost) AS google_ads_cost,
  SUM(google_ads_clicks) AS google_ads_clicks,
  SUM(google_ads_impressions) AS google_ads_impressions,
  SUM(google_ads_conversion_value) AS google_ads_conversion_value,
  SUM(google_ads_conversions) AS google_ads_conversions,

  COUNT(*) AS google_ads_source_row_count
FROM cleaned
GROUP BY 1,2;

**GA4**

In [ ]:
CREATE OR REPLACE TABLE `XXXX` AS
WITH source AS (
  SELECT
    SAFE_CAST(Item_ID___GA4__Google_Analytics AS STRING) AS Item_ID,
    SAFE_CAST(Item_revenue___GA4__Google_Analytics AS NUMERIC) AS Item_revenue,
    SAFE_CAST(Items_purchased___GA4__Google_Analytics AS INT64) AS Items_purchased,
    SAFE_CAST(Total_revenue___GA4__Google_Analytics AS INT64) AS Total_revenue,
    SAFE_CAST(Date AS DATE) AS Date,
    SAFE_CAST(Traffic_source AS STRING) AS Traffic_source,
    SAFE_CAST(Campaign AS STRING) AS Campaign,
    SAFE_CAST(Media_type AS STRING) AS Media_type,
    SAFE_CAST(Paid__Organic AS STRING) AS Paid_organic,
    SAFE_CAST(Session_medium___GA4__Google_Analytics AS STRING) AS Session_medium,
    SAFE_CAST(Ad_source___GA4__Google_Analytics AS STRING) AS Ad_source,
    SAFE_CAST(Session_campaign_ID___GA4__Google_Analytics AS STRING) AS Campaign_ID,
    SAFE_CAST(Source__medium___GA4__Google_Analytics AS STRING) AS Source_medium
  FROM `XXXX`
),

cleaned AS (
  SELECT
    Date,
    Item_revenue,
    Items_purchased,
    Total_revenue,
    Traffic_source,
    Campaign,
    Media_type,
    Paid_organic,
    Session_medium,
    Ad_source,
    Campaign_ID,
    Source_medium,
    CASE
      WHEN sku_base IN ('2360159', '2510433', '2810567') THEN CONCAT(sku_base, '11')
      ELSE sku_base
    END AS Item_ID_clean
  FROM (
    SELECT
      Date,
      COALESCE(Item_revenue,0) AS Item_revenue,
      COALESCE(Items_purchased,0) AS Items_purchased,
      COALESCE(Total_revenue,0) AS Total_revenue,
      Traffic_source,
      Campaign,
      Media_type,
      Paid_organic,
      Session_medium,
      Ad_source,
      Campaign_ID,
      Source_medium,
      REPLACE(
        REPLACE(Item_ID, '-', ''),
        'AVGBVQ25202NV',
        'AVGBVQ25202NVLF'
      ) AS sku_base
    FROM XXXX
  )
)

SELECT
  Item_ID_clean,
  Date,
  Source_medium,
  ANY_VALUE(Campaign_ID) AS Campaign_ID,
  ROUND(SUM(Item_revenue),2) AS Item_revenue,
  SUM(Items_purchased) AS Items_purchased,
  ROUND(SUM(Total_revenue),2) AS Total_revenue,
  ANY_VALUE(Traffic_source) AS Traffic_source,
  ANY_VALUE(Ad_source) AS Ad_source,
  ANY_VALUE(Media_type) AS Media_type,
  ANY_VALUE(Session_medium) AS Session_medium,
  ANY_VALUE(Paid_organic) AS Paid_organic,
FROM cleaned
GROUP BY 1,2,3;

**Join Process**

In [ ]:
CREATE OR REPLACE TABLE `Project_X511_S` AS
WITH ga4 AS (
  SELECT
    CAST(Item_ID_clean AS STRING) AS Item_ID_clean,
    CAST(Ad_source AS STRING) AS Ad_source,
    CAST(Campaign_ID AS STRING) AS Campaign_ID,
    SAFE_CAST(Date AS DATE) AS Date,
    SAFE_CAST(Item_revenue AS NUMERIC) AS Item_revenue,
    SAFE_CAST(Items_purchased AS NUMERIC) AS Items_purchased,
    SAFE_CAST(Total_revenue AS NUMERIC) AS Total_revenue_ga4,
    CAST(Traffic_source AS STRING) AS ga4_traffic_source,
    CAST(Media_type AS STRING) AS ga4_media_type,
    CAST(Paid_organic AS STRING) AS ga4_paid_organic,
    CAST(Session_medium AS STRING) AS Session_medium,
    CAST(Source_medium AS STRING) AS Source_medium
  FROM `X511_C`
),

ga4_agg AS (
  SELECT
    Item_ID_clean,
    Ad_source,
    Campaign_ID,
    Date,
    Source_medium,

    ANY_VALUE(ga4_traffic_source) AS ga4_traffic_source,
    ANY_VALUE(ga4_media_type) AS ga4_media_type,
    ANY_VALUE(ga4_paid_organic) AS ga4_paid_organic,
    ANY_VALUE(Session_medium) AS Session_medium,

    SUM(COALESCE(Item_revenue, 0)) AS Item_revenue,
    SUM(COALESCE(Items_purchased, 0)) AS Items_purchased,
    SUM(COALESCE(Total_revenue_ga4, 0)) AS Total_revenue_ga4
  FROM ga4
  GROUP BY
    Item_ID_clean,
    Ad_source,
    Campaign_ID,
    Date,
    Source_medium
),

sku AS (
  SELECT
    CAST(canonical_sku AS STRING) AS canonical_sku,
    SAFE_CAST(nett_cost AS NUMERIC) AS nett_cost,
    SAFE_CAST(source_row_count AS INT64) AS source_row_count,
    SAFE_CAST(distinct_name_count AS INT64) AS distinct_name_count
  FROM `X511_P`
),

google_ads_agg AS (
  SELECT
    CAST(campaign_id AS STRING) AS campaign_id,
    SAFE_CAST(Date AS DATE) AS Date,
    ANY_VALUE(campaign_name) AS campaign_name,
    ANY_VALUE(Traffic_source) AS google_ads_traffic_source,
    ANY_VALUE(paid_organic) AS google_ads_paid_organic,
    ANY_VALUE(Media_type) AS google_ads_media_type,
    ANY_VALUE(currency) AS currency,
    SUM(COALESCE(google_ads_clicks, 0)) AS google_ads_clicks,
    ROUND(SUM(COALESCE(google_ads_cost, 0)), 2) AS google_ads_cost,
    SUM(COALESCE(google_ads_impressions, 0)) AS google_ads_impressions,
    SUM(COALESCE(google_ads_conversions, 0)) AS google_ads_conversions
  FROM `X511_D`
  GROUP BY 1,2
),

matched_base AS (
  SELECT
    g.Item_ID_clean,
    g.Ad_source,
    g.Campaign_ID,
    g.Date,

    g.Item_revenue,
    g.Items_purchased,
    g.Total_revenue_ga4,
    g.ga4_traffic_source,
    g.ga4_media_type,
    g.ga4_paid_organic,
    g.Session_medium,
    g.Source_medium,

    a.campaign_name,
    a.google_ads_traffic_source,
    a.google_ads_paid_organic,
    a.google_ads_media_type,
    a.currency,
    a.google_ads_clicks,
    a.google_ads_cost,
    a.google_ads_impressions,
    a.google_ads_conversions,

    s.canonical_sku AS sku_id,
    s.nett_cost,
    s.source_row_count,
    s.distinct_name_count,

    CASE WHEN a.campaign_id IS NULL THEN 1 ELSE 0 END AS flag_missing_google_ads,
    CASE WHEN s.canonical_sku IS NULL THEN 1 ELSE 0 END AS flag_missing_sku
  FROM ga4_agg g
  LEFT JOIN google_ads_agg a
    ON g.Campaign_ID = a.campaign_id
   AND g.Date = a.Date
  LEFT JOIN sku s
    ON g.Item_ID_clean = s.canonical_sku
),

matched_with_shares AS (
  SELECT
    *,
    SUM(Item_revenue) OVER (
      PARTITION BY Date, Campaign_ID, Source_medium
    ) AS campaign_day_source_medium_total_revenue,

    SUM(Items_purchased) OVER (
      PARTITION BY Date, Campaign_ID, Source_medium
    ) AS campaign_day_source_medium_total_items
  FROM matched_base
),

matched_final AS (
  SELECT
    Date,
    Campaign_ID,
    campaign_name,

    Item_ID_clean,
    sku_id,

    Ad_source,
    ga4_traffic_source,
    ga4_media_type,
    ga4_paid_organic,
    Session_medium,
    Source_medium,

    google_ads_traffic_source,
    google_ads_paid_organic,
    google_ads_media_type,
    currency,

    Item_revenue,
    Items_purchased,
    nett_cost,
    Total_revenue_ga4 AS Total_revenue,
    SAFE_MULTIPLY(Items_purchased, nett_cost) AS total_nett_cost,

    google_ads_cost,
    google_ads_clicks,
    google_ads_impressions,
    google_ads_conversions,

    CASE
      WHEN campaign_day_source_medium_total_revenue > 0
        THEN SAFE_DIVIDE(Item_revenue, campaign_day_source_medium_total_revenue)
      ELSE 0
    END AS revenue_share,

    CASE
      WHEN campaign_day_source_medium_total_items > 0
        THEN SAFE_DIVIDE(Items_purchased, campaign_day_source_medium_total_items)
      ELSE 0
    END AS item_share,

    SAFE_MULTIPLY(
      google_ads_cost,
      CASE
        WHEN campaign_day_source_medium_total_revenue > 0
          THEN SAFE_DIVIDE(Item_revenue, campaign_day_source_medium_total_revenue)
        ELSE 0
      END
    ) AS allocated_cost,

    SAFE_MULTIPLY(
      google_ads_clicks,
      CASE
        WHEN campaign_day_source_medium_total_revenue > 0
          THEN SAFE_DIVIDE(Item_revenue, campaign_day_source_medium_total_revenue)
        ELSE 0
      END
    ) AS allocated_clicks,

    SAFE_MULTIPLY(
      google_ads_impressions,
      CASE
        WHEN campaign_day_source_medium_total_revenue > 0
          THEN SAFE_DIVIDE(Item_revenue, campaign_day_source_medium_total_revenue)
        ELSE 0
      END
    ) AS allocated_impressions,

    SAFE_MULTIPLY(
      google_ads_conversions,
      CASE
        WHEN campaign_day_source_medium_total_revenue > 0
          THEN SAFE_DIVIDE(Item_revenue, campaign_day_source_medium_total_revenue)
        ELSE 0
      END
    ) AS allocated_conversions,

    SAFE_SUBTRACT(
      Item_revenue,
      SAFE_MULTIPLY(Items_purchased, nett_cost)
    ) AS item_profit_before_ads,

    SAFE_SUBTRACT(
      SAFE_SUBTRACT(
        Item_revenue,
        SAFE_MULTIPLY(Items_purchased, nett_cost)
      ),
      SAFE_MULTIPLY(
        google_ads_cost,
        CASE
          WHEN campaign_day_source_medium_total_revenue > 0
            THEN SAFE_DIVIDE(Item_revenue, campaign_day_source_medium_total_revenue)
          ELSE 0
        END
      )
    ) AS item_profit_after_ads,

    0 AS flag_unmatched_spend_row,
    flag_missing_google_ads,
    flag_missing_sku
  FROM matched_with_shares
),

ga4_keys AS (
  SELECT DISTINCT
    Campaign_ID,
    Date
  FROM ga4_agg
),

unmatched_spend AS (
  SELECT
    a.Date,
    a.campaign_id AS Campaign_ID,
    a.campaign_name,

    '__UNMATCHED_SPEND__' AS Item_ID_clean,
    '__UNMATCHED_SPEND__' AS sku_id,

    CAST(NULL AS STRING) AS Ad_source,
    CAST(NULL AS STRING) AS ga4_traffic_source,
    CAST(NULL AS STRING) AS ga4_media_type,
    CAST(NULL AS STRING) AS ga4_paid_organic,
    CAST(NULL AS STRING) AS Session_medium,
    CAST(NULL AS STRING) AS Source_medium,

    a.google_ads_traffic_source,
    a.google_ads_paid_organic,
    a.google_ads_media_type,
    a.currency,

    CAST(0 AS NUMERIC) AS Item_revenue,
    CAST(0 AS NUMERIC) AS Items_purchased,
    CAST(0 AS NUMERIC) AS nett_cost,
    CAST(0 AS NUMERIC) AS Total_revenue,
    CAST(0 AS NUMERIC) AS total_nett_cost,

    a.google_ads_cost,
    a.google_ads_clicks,
    a.google_ads_impressions,
    a.google_ads_conversions,

    CAST(0 AS NUMERIC) AS revenue_share,
    CAST(0 AS NUMERIC) AS item_share,

    a.google_ads_cost AS allocated_cost,
    a.google_ads_clicks AS allocated_clicks,
    a.google_ads_impressions AS allocated_impressions,
    a.google_ads_conversions AS allocated_conversions,

    CAST(0 AS NUMERIC) AS item_profit_before_ads,
    SAFE_SUBTRACT(CAST(0 AS NUMERIC), a.google_ads_cost) AS item_profit_after_ads,

    1 AS flag_unmatched_spend_row,
    0 AS flag_missing_google_ads,
    0 AS flag_missing_sku
  FROM google_ads_agg a
  LEFT JOIN ga4_keys g
    ON a.campaign_id = g.Campaign_ID
   AND a.Date = g.Date
  WHERE g.Campaign_ID IS NULL
)

SELECT *
FROM matched_final

UNION ALL

SELECT *
FROM unmatched_spend;